# Full-sample normalization leak
The lagged signal is normalized using statistics from the entire sample.

In [ ]:
import numpy as np
import pandas as pd

def signal(prices):
    """Lag returns, but leak future observations through global z-score statistics."""
    lagged = prices.pct_change().shift(1)
    full_mean = lagged.mean(axis=0)
    full_std = lagged.std(axis=0).replace(0.0, np.nan)
    return ((lagged - full_mean) / full_std).fillna(0.0)

def backtest(prices):
    positions = signal(prices)
    asset_returns = prices.pct_change().fillna(0.0)
    return (positions * asset_returns).mean(axis=1)

def synthetic_prices():
    rng = np.random.default_rng(42)
    returns = rng.normal(0.001, 0.01, size=(120, 4))
    return pd.DataFrame(100.0 * (1.0 + returns).cumprod(axis=0), columns=list("ABCD"))
